# Notebook 17: Multi-Agent Customer Support

## Part 1: Multi-Agent Architecture, Shared State, and Agent Schemas


#### 1. Notebook purpose

This notebook extends the single-agent customer-support system into a
multi-agent architecture.

The system uses a coordinator agent to delegate work to specialized agents:

- SQL Agent
- Prediction Agent
- Vector Search Agent
- Retention Agent
- Final Response Agent

Part 1 establishes the foundation required for agent collaboration:

1. Define the multi-agent architecture.
2. Define each agent's responsibility.
3. Define validated input and output schemas.
4. Define a shared workflow state.
5. Establish consistent status and error-handling conventions.

The agents themselves will be implemented incrementally in later sections.

#### 2. Technologies Used

- Python
- Pydantic
- Python `TypedDict`
- Python `Literal`
- Databricks notebooks
- Existing customer-support tools from Notebooks 00–16
- Structured agent communication
- Shared workflow state
- Schema validation


#### 3: Multi-Agent Architecture

The multi-agent system separates planning, specialized execution, retention
decision-making, and final response generation.

```text

                         User Question
                               │
                               ▼
                      Coordinator Agent
                               │
                  Creates execution plan
                               │
       ┌───────────────┬──────────────────┬──────────────────┐
       ▼               ▼                  ▼                  │
   SQL Agent    Prediction Agent   Vector Search Agent       │
       │               │                  │                  │
       │               └──────────┬───────┘                  │
       │                          ▼                          │
       │                  Retention Agent                    │
       └──────────────────────────┴──────────────────────────┘
                               │
                               ▼
                    Final Response Agent
                               │
                               ▼
                       Grounded Answer

```

#### 4 : Agent Responsibilities

Each agent has one primary responsibility.

| Agent                | Responsibility                                                     |
| -------------------- | ------------------------------------------------------------------ |
| Coordinator Agent    | Analyze the customer's request and create an execution plan        |
| SQL Agent            | Retrieve structured business data required to answer the request   |
| Prediction Agent     | Classify the customer issue or request                             |
| Vector Search Agent  | Retrieve semantically similar historical support cases             |
| Retention Agent      | Recommend an appropriate customer retention action when applicable |
| Final Response Agent | Generate the final grounded response using validated agent results |


##### Design Principle

The coordinator delegates work but does not perform specialist tasks.

The final response agent presents results but does not call specialist tools.

#### 5 : Define imports


In [0]:
from typing import Any, Dict, List, Literal, Optional, TypedDict

from pydantic import BaseModel, ConfigDict, Field, ValidationError

In [0]:
# Why use ConfigDict(extra="forbid")? - It prevents agents from returning fields that are not part of the defined contract.
# Strict schemas make agent communication easier to debug and maintain.

{
    "agent_name": "prediction_agent",
    "status": "success",
    "predicted_category": "Technical",
    "unapproved_field": "unexpected value"
}

#### 6 : Define shared type aliases

In [0]:
# Why define controlled values? - Without controlled values, different agents might return inconsistent names.
# With Literal, the system uses one standard value:


AgentName = Literal[
    "coordinator_agent",
    "sql_agent",
    "prediction_agent",
    "vector_search_agent",
    "retention_agent",
    "final_response_agent",
]

AgentStatus = Literal[
    "pending",
    "running",
    "success",
    "failed",
    "skipped",
]

SupportCategory = Literal[
    "Billing",
    "Technical",
    "Service",
    "Account",
    "General",
]

RetentionAction = Literal[
    "offer_discount",
    "offer_support_package",
    "service_quality_review",
    "billing_review",
    "no_action",
]

#### 7 : Base agent result schema

In [0]:
#Every specialist agent should return common fields.

class BaseAgentResult(BaseModel):
    """
    Common fields returned by every agent.
    """

    model_config = ConfigDict(extra="forbid")

    agent_name: AgentName
    status: AgentStatus
    message: str = Field(
        description="Short description of the agent execution result."
    )
    error: Optional[str] = Field(
        default=None,
        description="Error details when agent execution fails.",
    )

In [0]:
# All agent outputs will contain:

{
    "agent_name": "...",
    "status": "...",
    "message": "...",
    "error": None
}

#This gives the orchestrator a consistent way to check every result.

#### 8 : Coordinator execution-plan schemas

- The Coordinator Agent is like a project manager. It doesn't solve the customer's problem directly—it creates a structured execution plan that tells the specialized agents what to do and in what order.

``` text

Customer Request
        │
        ▼
Coordinator Agent
        │
        ▼
Project Plan (CoordinatorResult)
        │
        ▼
Specialized Agents execute the plan
        │
        ▼
Final Response Agent

```

In [0]:
# The coordinator does not directly return a business answer. It returns a plan.
# First, define one task in the plan.

class AgentTask(BaseModel):
    """
    One task assigned by the coordinator to a specialized agent.
    """

    model_config = ConfigDict(extra="forbid")

    agent_name: Literal[
        "sql_agent",
        "prediction_agent",
        "vector_search_agent",
        "retention_agent",
    ]

    task_description: str = Field(
        min_length=1,
        description="Focused instruction for the selected agent.",
    )

    depends_on: List[AgentName] = Field(
        default_factory=list, 
        description="Agents that must complete before this task executes.",
    )

    # Create a brand new empty list every time a new object is created.As default value is a mutable object [], {}, set() then prefer: Field(default_factory=list) or Field(default_factory=dict)


In [0]:
# Now define the full coordinator result.

class CoordinatorResult(BaseAgentResult):
    """
    Structured execution plan created by the coordinator.
    """

    agent_name: Literal["coordinator_agent"] = "coordinator_agent"

    request_type: Literal[
        "analytics",
        "classification",
        "similarity_search",
        "retention",
        "combined",
        "general",
    ]

    tasks: List[AgentTask] = Field(
        default_factory=list,
        description="Ordered specialized-agent tasks.",
    )

    reasoning_summary: str = Field(
        description=(
            "Brief routing explanation without hidden chain-of-thought."
        )
    )

In [0]:
# Example coordinator result

coordinator_example = CoordinatorResult(
    status="success",
    message="Execution plan created successfully.",
    request_type="retention",
    tasks=[
        AgentTask(
            agent_name="prediction_agent",
            task_description="Predict the category of the customer issue.",
        ),
        AgentTask(
            agent_name="vector_search_agent",
            task_description="Retrieve similar historical support tickets.",
        ),
        AgentTask(
            agent_name="retention_agent",
            task_description="Recommend an appropriate retention action.",
            depends_on=[
                "prediction_agent",
                "vector_search_agent",
            ],
        ),
    ],
    reasoning_summary=(
        "The request requires issue classification, similar-ticket context, "
        "and a retention recommendation."
    ),
)

print(coordinator_example.model_dump())

In [0]:
#Expected structure:

{
    "agent_name": "coordinator_agent",
    "status": "success",
    "message": "Execution plan created successfully.",
    "error": None,
    "request_type": "retention",
    "tasks": [
        {
            "agent_name": "prediction_agent",
            "task_description": "Predict the category of the customer issue.",
            "depends_on": []
        },
        {
            "agent_name": "vector_search_agent",
            "task_description": "Retrieve similar historical support tickets.",
            "depends_on": []
        },
        {
            "agent_name": "retention_agent",
            "task_description": "Recommend an appropriate retention action.",
            "depends_on": [
                "prediction_agent",
                "vector_search_agent"
            ]
        }
    ],
    "reasoning_summary": "..."
}

#### 9 : SQL Agent result schema

In [0]:
# The SQL Agent should return structured analytics data.

class SQLAgentResult(BaseAgentResult):
    """
    Validated result returned by the SQL Agent.
    """

    agent_name: Literal["sql_agent"] = "sql_agent"

    sql_action: Optional[str] = Field(
        default=None,
        description="Name of the SQL analytics action that was executed.",
    )

    query_result: List[Dict[str, Any]] = Field(
        default_factory=list,
        description="Structured rows returned by the SQL tool.",
    )

    row_count: int = Field(
        default=0,
        ge=0,
        description="Number of rows returned.",
    )

In [0]:
sql_example = SQLAgentResult(
    status="success",
    message="Churned-customer count retrieved successfully.",
    sql_action="count_churned_customers",
    query_result=[
        {
            "churned_customers": 1869
        }
    ],
    row_count=1,
)

print(sql_example.model_dump())

#### 10 : Prediction Agent result schema

In [0]:
class PredictionAgentResult(BaseAgentResult):
    """
    Validated result returned by the Prediction Agent.
    """

    agent_name: Literal["prediction_agent"] = "prediction_agent"

    predicted_category: Optional[SupportCategory] = None

    confidence: Optional[float] = Field(
        default=None,
        ge=0.0,
        le=1.0,
        description="Prediction confidence when available.",
    )

    input_text: Optional[str] = Field(
        default=None,
        description="Customer-support text used for prediction.",
    )

    # What do ge and le mean? - These come from Pydantic validation. 
    # ge = 0.0 --> Greater than or equal to 0.0. le = 1.0 --> Less than or equal to 1.0. 
    # So the confidence must satisfy: 0.0 ≤ confidence ≤ 1.0
    # Confidence means how confident the model is in its prediction.
    # Why between 0 and 1? --> Most machine learning libraries represent confidence as a probability.

In [0]:
PredictionAgentResult(
    status="success",
    message="Prediction completed successfully.",
    predicted_category="Technical",
    confidence=None,
    input_text="Video streaming keeps buffering."
)

#### 11 : Vector Search Agent schemas

In [0]:
class SimilarTicket(BaseModel):
    """
    One support ticket retrieved through vector search.
    """

    model_config = ConfigDict(extra="forbid")

    ticket_id: str
    issue_text: str
    category: Optional[SupportCategory] = None

    similarity_score: Optional[float] = Field(
        default=None,
        ge=0.0,
        le=1.0,
    )

In [0]:
class VectorSearchAgentResult(BaseAgentResult):
    """
    Validated result returned by the Vector Search Agent.
    """

    agent_name: Literal["vector_search_agent"] = "vector_search_agent"

    query_text: Optional[str] = None

    similar_tickets: List[SimilarTicket] = Field(
        default_factory=list
    )

    result_count: int = Field(
        default=0,
        ge=0,
    )

In [0]:
vector_example = VectorSearchAgentResult(
    status="success",
    message="Similar support tickets retrieved successfully.",
    query_text="Internet is slow.",
    similar_tickets=[
        SimilarTicket(
            ticket_id="T010",
            issue_text="Internet connection is very slow.",
            category="Technical",
            similarity_score=0.94,
        ),
        SimilarTicket(
            ticket_id="T011",
            issue_text="Customer reports slow browsing speeds.",
            category="Technical",
            similarity_score=0.89,
        ),
        SimilarTicket(
            ticket_id="T017",
            issue_text="Network performance is inconsistent.",
            category="Technical",
            similarity_score=0.85,
        ),
    ],
    result_count=3,
)

print(vector_example.model_dump())

#### 12 :  Retention Agent result schema

In [0]:
#The Retention Agent consumes upstream context and returns a recommendation.

class RetentionAgentResult(BaseAgentResult):
    """
    Validated result returned by the Retention Agent.
    """

    agent_name: Literal["retention_agent"] = "retention_agent"

    recommended_action: Optional[RetentionAction] = None

    recommendation: Optional[str] = Field(
        default=None,
        description="Business-facing retention recommendation.",
    )

    reason: Optional[str] = Field(
        default=None,
        description="Brief grounded explanation for the recommendation.",
    )

    evidence_agents: List[AgentName] = Field(
        default_factory=list,
        description="Upstream agents whose results supported the decision.",
    )

In [0]:
retention_example = RetentionAgentResult(
    status="success",
    message="Retention recommendation created successfully.",
    recommended_action="offer_support_package",
    recommendation=(
        "Provide complimentary technical support or priority support."
    ),
    reason=(
        "The predicted category is Technical, and similar tickets indicate "
        "recurring service-quality concerns."
    ),
    evidence_agents=[
        "prediction_agent",
        "vector_search_agent",
    ],
)

print(retention_example.model_dump())

#### 13 : Final Response Agent result schema

In [0]:
class FinalResponseAgentResult(BaseAgentResult):
    """
    Validated result returned by the Final Response Agent.
    """

    agent_name: Literal["final_response_agent"] = "final_response_agent"

    final_answer: Optional[str] = Field(
        default=None,
        description="Grounded response returned to the user.",
    )

    source_agents: List[AgentName] = Field(
        default_factory=list,
        description="Agents whose results were used in the final response.",
    )

In [0]:
final_response_example = FinalResponseAgentResult(
    status="success",
    message="Final response generated successfully.",
    final_answer=(
        "Recommended Action: Provide complimentary technical support or "
        "priority support.\n\n"
        "Predicted Category: Technical\n\n"
        "Similar Tickets: T010, T011, T017"
    ),
    source_agents=[
        "prediction_agent",
        "vector_search_agent",
        "retention_agent",
    ],
)

print(final_response_example.model_dump())

#### 14 : Define the shared workflow state

The shared state is the central object passed through the orchestration workflow.

Each agent:

- Reads only the fields it needs.
- Performs its assigned work.
- Writes its validated result.
- Leaves unrelated fields unchanged.

In [0]:
class MultiAgentState(TypedDict):
    """
    Shared state passed through the multi-agent workflow.
    """

    # Original request
    user_question: str

    # Workflow identifiers and status
    workflow_id: str
    workflow_status: AgentStatus

    # Coordinator output
    coordinator_result: Optional[CoordinatorResult]

    # Specialist-agent outputs
    sql_result: Optional[SQLAgentResult]
    prediction_result: Optional[PredictionAgentResult]
    vector_search_result: Optional[VectorSearchAgentResult]
    retention_result: Optional[RetentionAgentResult]

    # Final output
    final_response_result: Optional[FinalResponseAgentResult]

    # Orchestration metadata
    completed_agents: List[AgentName]
    failed_agents: List[AgentName]
    execution_log: List[str]

#### 15 : Understand how shared state changes

In [0]:
# Initial state:

{
    "user_question": "Video streaming keeps buffering. What should we do?",
    "workflow_id": "workflow-001",
    "workflow_status": "pending",
    "coordinator_result": None,
    "sql_result": None,
    "prediction_result": None,
    "vector_search_result": None,
    "retention_result": None,
    "final_response_result": None,
    "completed_agents": [],
    "failed_agents": [],
    "execution_log": []
}

In [0]:
# After the coordinator:
    
{
    ...
    "workflow_status": "running",
    "coordinator_result": CoordinatorResult(...),
    "completed_agents": ["coordinator_agent"],
    "execution_log": [
        "Coordinator created a three-agent execution plan."
    ]
}

In [0]:
# After prediction:

{
    ...
    "prediction_result": PredictionAgentResult(
        predicted_category="Technical",
        ...
    ),
    "completed_agents": [
        "coordinator_agent",
        "prediction_agent"
    ]
}

In [0]:
# After vector search:
{
    ...
    "vector_search_result": VectorSearchAgentResult(...),
    "completed_agents": [
        "coordinator_agent",
        "prediction_agent",
        "vector_search_agent"
    ]
}

In [0]:
# Finally:
{
    ...
    "workflow_status": "success",
    "final_response_result": FinalResponseAgentResult(...),
    "completed_agents": [
        "coordinator_agent",
        "prediction_agent",
        "vector_search_agent",
        "retention_agent",
        "final_response_agent"
    ]
}

#The state becomes the system’s shared working context.

#### 16 : Create an initial-state function

In [0]:
from uuid import uuid4


def create_initial_state(user_question: str) -> MultiAgentState:
    """
    Create a clean initial state for one multi-agent workflow.

    Parameters
    ----------
    user_question:
        The original question submitted by the user.

    Returns
    -------
    MultiAgentState
        Initialized workflow state.
    """

    cleaned_question = user_question.strip()

    if not cleaned_question:
        raise ValueError("user_question must not be empty.")

    workflow_id = str(uuid4())

    return MultiAgentState(
        user_question=cleaned_question,
        workflow_id=workflow_id,
        workflow_status="pending",
        coordinator_result=None,
        sql_result=None,
        prediction_result=None,
        vector_search_result=None,
        retention_result=None,
        final_response_result=None,
        completed_agents=[],
        failed_agents=[],
        execution_log=[
            f"Workflow {workflow_id} initialized."
        ],
    )

In [0]:
#test it:
state = create_initial_state(
    "Video streaming keeps buffering. What should we do?"
)

state

In [0]:
#### 17 : Add a safe state-update helper
#We want consistent updates to completed_agents, failed_agents, and the execution log.

def record_agent_execution(
    state: MultiAgentState,
    agent_name: AgentName,
    status: AgentStatus,
    message: str,
) -> MultiAgentState:
    """
    Record an agent's execution status in the shared workflow state.
    """

    if status == "success":
        if agent_name not in state["completed_agents"]:
            state["completed_agents"].append(agent_name)

    elif status == "failed":
        if agent_name not in state["failed_agents"]:
            state["failed_agents"].append(agent_name)

    state["execution_log"].append(
        f"{agent_name}: {status} - {message}"
    )

    return state

In [0]:
state = record_agent_execution(
    state=state,
    agent_name="prediction_agent",
    status="success",
    message="Predicted category: Technical",
)

state["completed_agents"]

#### 18 : Validate schema behavior

In [0]:
valid_prediction = PredictionAgentResult(
    status="success",
    message="Prediction completed.",
    predicted_category="Technical",
    confidence=0.88,
    input_text="Internet is slow.",
)

print(valid_prediction.model_dump())

In [0]:
# Invalid category
#Pydantic should reject "UnknownCategory" because it is not in the support category.
try:
    PredictionAgentResult(
        status="success",
        message="Prediction completed.",
        predicted_category="UnknownCategory",
        confidence=0.88,
        input_text="Internet is slow.",
    )

except ValidationError as exc:
    print("VALIDATION ERROR")
    print(exc)

In [0]:
# Invalid confidence
try:
    PredictionAgentResult(
        status="success",
        message="Prediction completed.",
        predicted_category="Technical",
        confidence=1.5,
        input_text="Internet is slow.",
    )

except ValidationError as exc:
    print("VALIDATION ERROR")
    print(exc)

In [0]:
# Unexpected field

try:
    PredictionAgentResult(
        status="success",
        message="Prediction completed.",
        predicted_category="Technical",
        confidence=0.88,
        input_text="Internet is slow.",
        unsupported_field="unexpected",
    )

except ValidationError as exc:
    print("VALIDATION ERROR")
    print(exc)

#### 19 : Add cross-field validation

In [0]:
#A failed agent should provide an error message. A successful agent should not contain an error.

from pydantic import model_validator


class BaseAgentResult(BaseModel):
    """
    Common fields returned by every agent.
    """

    model_config = ConfigDict(extra="forbid")

    agent_name: AgentName
    status: AgentStatus

    message: str = Field(
        description="Short description of the agent execution result."
    )

    error: Optional[str] = Field(
        default=None,
        description="Error details when agent execution fails.",
    )

    @model_validator(mode="after")
    def validate_status_and_error(self):
        """
        Enforce consistent success and failure behavior.
        """

        if self.status == "failed" and not self.error:
            raise ValueError(
                "A failed agent result must include an error message."
            )

        if self.status == "success" and self.error is not None:
            raise ValueError(
                "A successful agent result must not include an error."
            )

        return self

In [0]:
#Test failure validation:

try:
    SQLAgentResult(
        status="failed",
        message="SQL execution failed.",
        sql_action="count_billing_tickets",
    )

except ValidationError as exc:
    print("VALIDATION ERROR")
    print(exc)

In [0]:
# Correct failed result:

failed_sql_result = SQLAgentResult(
    status="failed",
    message="SQL execution failed.",
    error="The SQL analytics tool returned an invalid response.",
    sql_action="count_billing_tickets",
    query_result=[],
    row_count=0,
)

print(failed_sql_result.model_dump())


#### 20 : Dependency examples

In [0]:
# Dependencies tell the orchestrator which results must exist before another agent executes.
# This means the orchestrator should not run the Retention Agent until both upstream agents succeed.

retention_task = AgentTask(
    agent_name="retention_agent",
    task_description="Recommend a customer retention action.",
    depends_on=[
        "prediction_agent",
        "vector_search_agent",
    ],
)

In [0]:
# However, an SQL question may have no specialist dependency:

sql_task = AgentTask(
    agent_name="sql_agent",
    task_description="Count Billing support tickets.",
    depends_on=[],
)

#### 21 : Architecture examples


##### Scenario 1: SQL analytics only

Question:

- How many Billing tickets are there?

Execution plan:

``` text

Coordinator Agent
      │
      ▼
   SQL Agent
      │
      ▼
Final Response Agent

```

In [0]:
AgentTask(
    agent_name="sql_agent",
    task_description="Count support tickets with category Billing.",
    depends_on=[],
)

##### Scenario 2: Prediction only

Question:

- What category is this issue: Video streaming keeps buffering?

Flow:

``` text

Coordinator Agent
      │
      ▼
Prediction Agent
      │
      ▼
Final Response Agent

```


##### Scenario 3: Similar-ticket search only

Question:

- Find tickets similar to: Internet is slow.

Flow:

``` text 

Coordinator Agent
      │
      ▼
Vector Search Agent
      │
      ▼
Final Response Agent

```


##### Scenario 4: Retention workflow

Question:

- Video streaming keeps buffering. What should we do to retain the customer?

Flow:

``` text

Coordinator Agent
      │
      ├───────────────┐
      ▼               ▼
Prediction Agent   Vector Search Agent
      │               │
      └───────┬───────┘
              ▼
       Retention Agent
              │
              ▼
    Final Response Agent

```

#### 22 : Recommended schema summary

The complete schema relationship is:

``` text

BaseAgentResult
│
├── CoordinatorResult
│       └── List[AgentTask]
│
├── SQLAgentResult
│
├── PredictionAgentResult
│
├── VectorSearchAgentResult
│       └── List[SimilarTicket]
│
├── RetentionAgentResult
│
└── FinalResponseAgentResult

```


#### 23 : Key learnings

1. A multi-agent system divides a complex workflow among specialized agents.

2. Each agent should have one clearly defined responsibility.

3. The coordinator creates an execution plan but does not perform specialist
   tasks itself.

4. Agent schemas create explicit communication contracts between agents.

5. Pydantic validates agent outputs before they are stored in shared state.

6. `TypedDict` describes the complete workflow state while allowing the state
   to grow during execution.

7. Shared state allows agents to exchange structured context without relying
   on unstructured conversational history.

8. Agent dependencies define execution order.

9. Structured results should remain separate from the final user-facing
   response.

10. Consistent status, error, and logging fields make multi-agent workflows
    easier to test and debug.

#### 24 : Part 1 conclusion

Part 1 established the foundation for the multi-agent customer-support system.

The architecture now contains:

- A coordinator responsible for delegation
- Specialized agent roles
- Standard agent names and statuses
- Validated agent-result schemas
- Dependency-aware agent tasks
- A shared workflow state
- Execution tracking and error conventions

No agent has been implemented yet. This is intentional.

Defining the communication contracts first ensures that every later agent
returns predictable and validated data.

In Part 2, we will implement the Coordinator Agent. It will analyze the user
question and produce a structured execution plan using the schemas defined in
Part 1.

## Part 2: Coordinator Agent

#### 1 : Purpose

The Coordinator Agent will:

- Read the customer request.
- Identify the request type.
- Select the required specialist agents.
- define task dependencies.
- Return a validated CoordinatorResult.
- Store the result in shared state.

The Coordinator will not:

- Query SQL tables.
- Predict ticket categories.
- Perform vector search.
- Recommend retention actions.
- Write the final customer-facing response.

Its responsibility is only planning.

#### 2 : Coordinator flow

``` text

Customer request
      │
      ▼
Coordinator Agent
      │
      ├── Identify request type
      │
      ├── Select agents
      │
      ├── Create tasks
      │
      └── Define dependencies
      │
      ▼
Validated execution plan
      │
      ▼
Shared state

```


#### 3 : Example routes

| Customer request                                | Required workflow                                       |
| ----------------------------------------------- | ------------------------------------------------------- |
| “How many Billing tickets are there?”           | SQL → Final Response                                    |
| “What category is this issue?”                  | Prediction → Final Response                             |
| “Find tickets similar to internet is slow.”     | Vector Search → Final Response                          |
| “Streaming keeps buffering. What should we do?” | Prediction + Vector Search → Retention → Final Response |


The Coordinator does not necessarily call every agent.

#### 4 : Imports

In [0]:
from typing import List, Literal

from pydantic import ValidationError


#### 5 : Define request types

In [0]:
#The request type describes the kind of workflow the Coordinator selected.

RequestType = Literal[
    "sql_analytics",
    "prediction",
    "similar_tickets",
    "retention",
    "general",
]


#### 6 : Request classification

In [0]:
def classify_request(user_request: str) -> RequestType:
    """
    Classify the user request into one supported workflow.

    This first implementation uses deterministic keyword rules.
    It can later be replaced by an LLM-based classifier.
    """

    normalized_request = user_request.strip().lower()

    if not normalized_request:
        return "general"

    retention_keywords = [
        "what should we do",
        "recommend",
        "retention",
        "retain",
        "offer",
        "action",
        "customer is unhappy",
        "customer may leave",
        "cancel",
        "cancellation",
    ]

    similar_ticket_keywords = [
        "similar ticket",
        "similar tickets",
        "similar case",
        "similar cases",
        "historical ticket",
        "related ticket",
        "find tickets like",
    ]

    prediction_keywords = [
        "predict",
        "prediction",
        "classify",
        "category",
        "ticket type",
        "issue type",
        "what type of issue",
    ]

    sql_keywords = [
        "how many",
        "count",
        "total number",
        "average",
        "percentage",
        "rate",
        "most common",
        "least common",
        "distribution",
        "show all",
        "list all",
    ]

    if any(keyword in normalized_request for keyword in retention_keywords):
        return "retention"

    if any(keyword in normalized_request for keyword in similar_ticket_keywords):
        return "similar_tickets"

    if any(keyword in normalized_request for keyword in prediction_keywords):
        return "prediction"

    if any(keyword in normalized_request for keyword in sql_keywords):
        return "sql_analytics"

    return "general"


#### 7 : Test request classification

In [0]:
classification_tests = [
    "How many Billing tickets are there?",
    "Predict the category for: Video streaming keeps buffering.",
    "Find tickets similar to internet is slow.",
    "The customer is unhappy and may cancel. What should we do?",
    "Hello, can you help me?",
]

for question in classification_tests:
    request_type = classify_request(question)

    print(f"Question: {question}")
    print(f"Request type: {request_type}")
    print("-" * 80)

#### 8 : Build SQL execution plan

In [0]:
def build_sql_plan(user_request: str) -> List[AgentTask]:
    return [
        AgentTask(
            agent_name="sql_agent",
            task_description=(
                "Analyze the structured support data and answer the "
                f"analytics request: {user_request}"
            ),
            depends_on=[],
        ),
        AgentTask(
            agent_name="final_response_agent",
            task_description=(
                "Create a grounded user-facing response from the SQL Agent result."
            ),
            depends_on=["sql_agent"],
        ),
    ]

#### 9 : Build Prediction execution plan

In [0]:
def build_prediction_plan(user_request: str) -> List[AgentTask]:
    return [
        AgentTask(
            agent_name="prediction_agent",
            task_description=(
                "Predict the support issue category for the following request: "
                f"{user_request}"
            ),
            depends_on=[],
        ),
        AgentTask(
            agent_name="final_response_agent",
            task_description=(
                "Create a grounded user-facing response from the "
                "Prediction Agent result."
            ),
            depends_on=["prediction_agent"],
        ),
    ]


#### 10 : Build Vector Search execution plan

In [0]:
def build_similar_tickets_plan(user_request: str) -> List[AgentTask]:
    return [
        AgentTask(
            agent_name="vector_search_agent",
            task_description=(
                "Retrieve semantically similar historical support cases "
                f"for the following request: {user_request}"
            ),
            depends_on=[],
        ),
        AgentTask(
            agent_name="final_response_agent",
            task_description=(
                "Create a grounded user-facing response using the "
                "Vector Search Agent result."
            ),
            depends_on=["vector_search_agent"],
        ),
    ]


#### 11 : Build Retention execution plan

In [0]:
def build_retention_plan(user_request: str) -> List[AgentTask]:
    return [
        AgentTask(
            agent_name="prediction_agent",
            task_description=(
                "Predict the support issue category for the following request: "
                f"{user_request}"
            ),
            depends_on=[],
        ),
        AgentTask(
            agent_name="vector_search_agent",
            task_description=(
                "Retrieve semantically similar historical support cases "
                f"for the following request: {user_request}"
            ),
            depends_on=[],
        ),
        AgentTask(
            agent_name="retention_agent",
            task_description=(
                "Recommend an appropriate retention action using the predicted "
                "category and similar historical support cases."
            ),
            depends_on=[
                "prediction_agent",
                "vector_search_agent",
            ],
        ),
        AgentTask(
            agent_name="final_response_agent",
            task_description=(
                "Create the final grounded user-facing response using the "
                "Prediction, Vector Search, and Retention Agent results."
            ),
            depends_on=[
                "prediction_agent",
                "vector_search_agent",
                "retention_agent",
            ],
        ),
    ]


#### 12 : Build General execution plan

In [0]:
def build_general_plan(user_request: str) -> List[AgentTask]:
    return [
        AgentTask(
            agent_name="final_response_agent",
            task_description=(
                "Respond to the general customer-support request without "
                f"calling specialist tools: {user_request}"
            ),
            depends_on=[],
        )
    ]

#### 13 : Create the execution plan

In [0]:
def create_execution_plan(
    user_request: str,
    request_type: RequestType,
) -> List[AgentTask]:
    """
    Create the ordered list of agent tasks for the selected request type.
    """

    if request_type == "sql_analytics":
        return build_sql_plan(user_request)

    if request_type == "prediction":
        return build_prediction_plan(user_request)

    if request_type == "similar_tickets":
        return build_similar_tickets_plan(user_request)

    if request_type == "retention":
        return build_retention_plan(user_request)

    return build_general_plan(user_request)


#### 14 : Coordinator Agent function

In [0]:
def coordinator_agent(user_request: str) -> CoordinatorResult:
    """
    Analyze a customer request and return a validated execution plan.

    The Coordinator only plans the workflow. It does not execute
    specialist tasks.
    """

    cleaned_request = user_request.strip()

    if not cleaned_request:
        return CoordinatorResult(
            agent_name="coordinator_agent",
            status="failed",
            message="The customer request cannot be empty.",
            error="EMPTY_USER_REQUEST",
            request_type="general",
            tasks=[],
        )

    request_type = classify_request(cleaned_request)

    tasks = create_execution_plan(
        user_request=cleaned_request,
        request_type=request_type,
    )

    return CoordinatorResult(
        agent_name="coordinator_agent",
        status="success",
        message=(
            f"Execution plan created successfully for request type: "
            f"{request_type}."
        ),
        error=None,
        request_type=request_type,
        tasks=tasks,
    )


# Because CoordinatorResult is a Pydantic model, the returned plan is validated automatically. For example, Pydantic can catch:
    # Unsupported agent names.
    # Unsupported status values.
    # Missing task descriptions.
    # Invalid dependency structures.
    # Incorrect field types.


#### 15 : Test the Coordinator

###### SQL request

In [0]:
sql_result = coordinator_agent(
    "How many Billing tickets are there?"
)

print(sql_result.model_dump())

In [0]:
# Expected structure:

{
    "agent_name": "coordinator_agent",
    "status": "success",
    "message": (
        "Execution plan created successfully for request type: "
        "sql_analytics."
    ),
    "error": None,
    "request_type": "sql_analytics",
    "tasks": [
        {
            "agent_name": "sql_agent",
            "task_description": "...",
            "depends_on": [],
        },
        {
            "agent_name": "final_response_agent",
            "task_description": "...",
            "depends_on": ["sql_agent"],
        },
    ],
}


###### Prediction request

In [0]:
prediction_result = coordinator_agent(
    "Predict the category for: Video streaming keeps buffering."
)

print(prediction_result.model_dump())


###### Similar-ticket request

In [0]:
vector_result = coordinator_agent(
    "Find tickets similar to internet is slow."
)

print(vector_result.model_dump())


###### Retention request

In [0]:
retention_result = coordinator_agent(
    "Video streaming keeps buffering and the customer may cancel. "
    "What should we recommend?"
)

print(retention_result.model_dump())


#### 16 : Store the plan in shared state

In [0]:
# Assuming your Part 1 shared state includes fields such as:

class MultiAgentState(TypedDict):
    user_request: str
    coordinator_result: CoordinatorResult | None
    agent_results: dict
    execution_history: list
    final_response: str | None
    errors: list

In [0]:
def run_coordinator(state: MultiAgentState) -> MultiAgentState:
    """
    Run the Coordinator Agent and store its result in shared state.
    """

    user_request = state["user_request"]

    coordinator_result = coordinator_agent(user_request)

    state["coordinator_result"] = coordinator_result

    state["execution_history"].append(
        {
            "agent_name": "coordinator_agent",
            "status": coordinator_result.status,
            "message": coordinator_result.message,
        }
    )

    if coordinator_result.status == "failed":
        state["errors"].append(
            {
                "agent_name": "coordinator_agent",
                "error": coordinator_result.error,
            }
        )

    return state

#### 17 : End-to-end Coordinator test

In [0]:
state = create_initial_state(
    user_request=(
        "Video streaming keeps buffering and the customer may cancel. "
        "What should we recommend?"
    )
)

updated_state = run_coordinator(state)

print("Request:")
print(updated_state["user_request"])

print("\nCoordinator result:")
print(updated_state["coordinator_result"].model_dump())

print("\nExecution history:")
print(updated_state["execution_history"])

print("\nErrors:")
print(updated_state["errors"])


#### 18 : Important distinction


These two functions have different responsibilities:

In [0]:
coordinator_agent(user_request)


- Performs Coordinator logic.
- Returns a validated CoordinatorResult.
- Does not know about the entire workflow state.

In [0]:
run_coordinator(state)


- Reads from shared state.
- Calls the Coordinator.
- Writes the result back to shared state.
- Records execution history and errors.
- 
This separation makes the Coordinator easier to unit test.

#### 19 : Recommended assertions

In [0]:
# Instead of only printing results, add assertions.

sql_plan = coordinator_agent(
    "How many Billing tickets are there?"
)

assert sql_plan.status == "success"
assert sql_plan.request_type == "sql_analytics"
assert len(sql_plan.tasks) == 2
assert sql_plan.tasks[0].agent_name == "sql_agent"
assert sql_plan.tasks[1].agent_name == "final_response_agent"
assert sql_plan.tasks[1].depends_on == ["sql_agent"]

In [0]:
# Retention test:

retention_plan = coordinator_agent(
    "The customer may cancel because streaming keeps buffering. "
    "What should we recommend?"
)

assert retention_plan.status == "success"
assert retention_plan.request_type == "retention"
assert len(retention_plan.tasks) == 4

assert retention_plan.tasks[0].agent_name == "prediction_agent"
assert retention_plan.tasks[1].agent_name == "vector_search_agent"
assert retention_plan.tasks[2].agent_name == "retention_agent"
assert retention_plan.tasks[3].agent_name == "final_response_agent"

assert retention_plan.tasks[2].depends_on == [
    "prediction_agent",
    "vector_search_agent",
]

In [0]:
#Empty request test:

empty_result = coordinator_agent("   ")

assert empty_result.status == "failed"
assert empty_result.error == "EMPTY_USER_REQUEST"
assert empty_result.tasks == []

#### 20 : Key learning


The Coordinator performs three distinct steps:

``` text

Request classification
        │
        ▼
Workflow selection
        │
        ▼
Execution-plan creation

```

The output of Part 2 is therefore:

``` text

Customer request
        │
        ▼
Coordinator Agent
        │
        ▼
Validated CoordinatorResult
        │
        ▼
Stored in shared state

```

The next specialist agent can read the execution plan and perform only the work assigned to it.


#### Part 2 conclusion

You now have a Coordinator Agent that:

- Identifies the workflow.
- Selects only the necessary agents.
- Creates task descriptions.
- Defines dependencies.
- Validates the plan with Pydantic.
- Records the plan in shared state.
- Handles empty requests safely.
- Can be tested independently.


## Part 3 – SQL Agent

The agents are not calling each other directly.Instead they communicate through the Shared State.

#### Purpose

- The SQL Agent is responsible for executing SQL analytics tasks assigned by the Coordinator Agent.

- It does not decide whether SQL should be used.

It simply:

- Reads the execution plan.
- Checks whether it has been assigned a task.
- Executes the SQL Analytics Tool.
- Validates the result.
- Updates the shared state.


#### Architecture


``` text

Shared State
      │
      ▼
Read Coordinator Result
      │
      ▼
Was SQL Agent assigned?
      │
   ┌──┴─────────┐
   │            │
  No           Yes
   │            │
   ▼            ▼
Record       Read assigned
Skipped      SQL task
                │
                ▼
       Call SQL Analytics Tool
                │
                ▼
       Validate SQLAgentResult
                │
                ▼
       Store in agent_results
                │
                ▼
       Record execution history
                │
                ▼
        Record error if needed


```

#### 1 : Imports

In [0]:
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from pydantic import BaseModel, Field, ValidationError


This assumes the following Part 1 types and schemas are already available:

- AgentName
- AgentStatus
- AgentTask
- CoordinatorResult
- SQLAgentResult
- MultiAgentState


#### 2 : Execution-history schema

In [0]:
class ExecutionRecord(BaseModel):
    """
    Represents one agent execution event in the multi-agent workflow.
    """

    agent_name: AgentName

    status: AgentStatus

    message: str

    timestamp: datetime = Field(
        default_factory=lambda: datetime.now(timezone.utc)
    )

In [0]:
# testing

ExecutionRecord(
    agent_name="sql_agent",
    status="success",
    message="SQL analytics task completed successfully.",
)


#### 3 : Updated shared-state definition

In [0]:
class MultiAgentState(TypedDict):
    """
    Shared state used by all agents in the workflow.
    """

    user_request: str

    coordinator_result: Optional[CoordinatorResult]

    agent_results: Dict[str, BaseAgentResult]

    execution_history: List[ExecutionRecord]

    final_response: Optional[str]

    errors: List[Dict[str, Any]]


#### 4 : Initial-state function

In [0]:
def create_initial_state(
    user_request: str,
) -> MultiAgentState:
    """
    Create the initial shared state for one customer request.
    """

    return {
        "user_request": user_request.strip(),
        "coordinator_result": None,
        "agent_results": {},
        "execution_history": [],
        "final_response": None,
        "errors": [],
    }

In [0]:
state = create_initial_state(
    "How many Billing tickets are there?"
)


#### 5 : Helper: find an assigned task

In [0]:
def get_agent_task(
    tasks: List[AgentTask],
    agent_name: AgentName,
) -> Optional[AgentTask]:
    """
    Return the task assigned to the requested agent.

    Parameters
    ----------
    tasks:
        Execution-plan tasks created by the Coordinator Agent.

    agent_name:
        Name of the specialist agent looking for its task.

    Returns
    -------
    Optional[AgentTask]
        The assigned task when found; otherwise None.
    """

    for task in tasks:
        if task.agent_name == agent_name:
            return task

    return None

In [0]:
sql_task = get_agent_task(
    tasks=coordinator_result.tasks,
    agent_name="sql_agent",
)

In [0]:
retention_task = get_agent_task(
    sql_result.tasks,
    "retention_agent",
)

print(retention_task)


#### 6 : Helper -  record agent execution

In [0]:
# standardize execution history

def record_agent_execution(
    state: MultiAgentState,
    agent_name: AgentName,
    status: AgentStatus,
    message: str,
) -> None:
    """
    Add one validated execution event to shared state.
    """

    execution_record = ExecutionRecord(
        agent_name=agent_name,
        status=status,
        message=message,
    )

    state["execution_history"].append(execution_record)

    # This helper modifies the shared-state dictionary directly, so it does not need to return the state.

In [0]:
state = create_initial_state(
    "How many Billing tickets?"
)

record_agent_execution(
    state,
    "sql_agent",
    "success",
    "SQL query executed successfully.",
)

print(state["execution_history"])

In [0]:
[
    {
        "agent_name": "sql_agent",
        "status": "success",
        "message": "SQL query executed successfully."
    }
]


#### 7 : Helper - record workflow errors

In [0]:
def record_agent_error(
    state: MultiAgentState,
    agent_name: AgentName,
    error_code: str,
    error_message: str,
) -> None:
    """
    Record one agent error in shared state.
    """

    state["errors"].append(
        {
            "agent_name": agent_name,
            "error_code": error_code,
            "error_message": error_message,
        }
    )

# This keeps error-handling code consistent across the specialist agents.


#### 8 : SQL Agent function

In [0]:
#The SQL Agent receives an assigned AgentTask and calls the existing SQL Analytics Tool.

def execute_sql_agent(
    task: AgentTask,
    user_request: str,
) -> SQLAgentResult:
    """
    Execute the SQL Analytics Tool for one assigned SQL task.

    The SQL Agent does not decide whether SQL is required.
    The Coordinator Agent has already made that decision.
    """

    try:
        tool_result = sql_analytics_tool(user_request)

        if not isinstance(tool_result, dict):
            return SQLAgentResult(
                agent_name="sql_agent",
                status="failed",
                message="The SQL Analytics Tool returned an invalid result.",
                error="INVALID_SQL_TOOL_RESULT",
                task_description=task.task_description,
                sql_action=None,
                sql_result=None,
                row_count=None,
            )

        tool_status = tool_result.get("status")

        if tool_status != "success":
            return SQLAgentResult(
                agent_name="sql_agent",
                status="failed",
                message="The SQL Analytics Tool could not complete the request.",
                error=tool_result.get(
                    "error",
                    "SQL_TOOL_EXECUTION_FAILED",
                ),
                task_description=task.task_description,
                sql_action=tool_result.get("sql_action"),
                sql_result=tool_result.get("sql_result"),
                row_count=None,
            )

        sql_result = tool_result.get("sql_result")

        row_count = (
            len(sql_result)
            if isinstance(sql_result, list)
            else None
        )

        return SQLAgentResult(
            agent_name="sql_agent",
            status="success",
            message="SQL analytics task completed successfully.",
            error=None,
            task_description=task.task_description,
            sql_action=tool_result.get("sql_action"),
            sql_result=sql_result,
            row_count=row_count,
        )

    except Exception as exc:
        return SQLAgentResult(
            agent_name="sql_agent",
            status="failed",
            message="An unexpected error occurred in the SQL Agent.",
            error=str(exc),
            task_description=task.task_description,
            sql_action=None,
            sql_result=None,
            row_count=None,
        )

In [0]:
{
    "tool": "sql_analytics_tool",
    "status": "success",
    "sql_action": "count_billing_tickets",
    "sql_result": [
        Row(billing_tickets=25)
    ],
}


#### 9 : Shared-state SQL runner

In [0]:
# This is the function used by the multi-agent workflow.

def run_sql_agent(
    state: MultiAgentState,
) -> MultiAgentState:
    """
    Run the SQL Agent only when the Coordinator assigned it a task.

    Workflow:
    1. Read Coordinator result.
    2. Find SQL Agent task.
    3. Execute SQL Analytics Tool.
    4. Validate the SQL Agent result.
    5. Store result in shared state.
    6. Record execution history and errors.
    """

    agent_name: AgentName = "sql_agent"

    coordinator_result = state.get("coordinator_result")

    if coordinator_result is None:
        message = (
            "SQL Agent could not run because the Coordinator result "
            "is missing."
        )

        failed_result = SQLAgentResult(
            agent_name="sql_agent",
            status="failed",
            message=message,
            error="MISSING_COORDINATOR_RESULT",
            task_description=None,
            sql_action=None,
            sql_result=None,
            row_count=None,
        )

        state["agent_results"][agent_name] = failed_result

        record_agent_execution(
            state=state,
            agent_name=agent_name,
            status="failed",
            message=message,
        )

        record_agent_error(
            state=state,
            agent_name=agent_name,
            error_code="MISSING_COORDINATOR_RESULT",
            error_message=message,
        )

        return state

    if coordinator_result.status != "success":
        message = (
            "SQL Agent could not run because the Coordinator Agent "
            "did not create a successful execution plan."
        )

        failed_result = SQLAgentResult(
            agent_name="sql_agent",
            status="failed",
            message=message,
            error="INVALID_COORDINATOR_RESULT",
            task_description=None,
            sql_action=None,
            sql_result=None,
            row_count=None,
        )

        state["agent_results"][agent_name] = failed_result

        record_agent_execution(
            state=state,
            agent_name=agent_name,
            status="failed",
            message=message,
        )

        record_agent_error(
            state=state,
            agent_name=agent_name,
            error_code="INVALID_COORDINATOR_RESULT",
            error_message=message,
        )

        return state

    sql_task = get_agent_task(
        tasks=coordinator_result.tasks,
        agent_name=agent_name,
    )

    if sql_task is None:
        record_agent_execution(
            state=state,
            agent_name=agent_name,
            status="skipped",
            message=(
                "SQL Agent was not assigned a task by the "
                "Coordinator Agent."
            ),
        )

        return state

    sql_result = execute_sql_agent(
        task=sql_task,
        user_request=state["user_request"],
    )

    state["agent_results"][agent_name] = sql_result

    record_agent_execution(
        state=state,
        agent_name=agent_name,
        status=sql_result.status,
        message=sql_result.message,
    )

    if sql_result.status == "failed":
        record_agent_error(
            state=state,
            agent_name=agent_name,
            error_code=sql_result.error or "SQL_AGENT_FAILED",
            error_message=sql_result.message,
        )

    return state


#### 10 :  Why use both functions?

In [0]:
execute_sql_agent(
    task,
    user_request,
)


- Executes the SQL-specific work.
- Calls the SQL tool.
- Returns SQLAgentResult.
- Does not manage the full shared state.

In [0]:
run_sql_agent(state)


- Reads the Coordinator plan.
- Determines whether SQL work was assigned.
- Calls execute_sql_agent().
- Stores the result.
- Updates execution history.
- Records errors.

This separation makes testing easier.


#### 11. Test 1: SQL task assigned

In [0]:
state = create_initial_state(
    "How many Billing tickets are there?"
)

state = run_coordinator(state)

state = run_sql_agent(state)

sql_result = state["agent_results"]["sql_agent"]

print("SQL Agent Result:")
print(sql_result.model_dump())

print("\nExecution History:")
for record in state["execution_history"]:
    print(record.model_dump())

print("\nErrors:")
print(state["errors"])

In [0]:
#Assertions:

assert state["coordinator_result"] is not None
assert state["coordinator_result"].request_type == "sql_analytics"

assert "sql_agent" in state["agent_results"]

assert sql_result.status == "success"
assert sql_result.agent_name == "sql_agent"
assert sql_result.sql_action is not None

assert state["errors"] == []


#### 12 : Test 2 -  SQL Agent not assigned

In [0]:
state = create_initial_state(
    "Predict the category for video streaming keeps buffering."
)

state = run_coordinator(state)

state = run_sql_agent(state)

print("Agent Results:")
print(state["agent_results"])

print("\nExecution History:")
for record in state["execution_history"]:
    print(record.model_dump())

In [0]:
assert state["coordinator_result"].request_type == "prediction"

assert "sql_agent" not in state["agent_results"]

sql_history = [
    record
    for record in state["execution_history"]
    if record.agent_name == "sql_agent"
]

assert len(sql_history) == 1
assert sql_history[0].status == "skipped"

# A skipped agent is not an error.


#### 13 -  Test 3 : Missing Coordinator result

In [0]:
state = create_initial_state(
    "How many Billing tickets are there?"
)

state = run_sql_agent(state)

sql_result = state["agent_results"]["sql_agent"]

print(sql_result.model_dump())
print(state["errors"])

In [0]:
assert sql_result.status == "failed"
assert sql_result.error == "MISSING_COORDINATOR_RESULT"

assert len(state["errors"]) == 1

assert (
    state["errors"][0]["error_code"]
    == "MISSING_COORDINATOR_RESULT"
)

#This confirms that specialist agents cannot execute without a valid Coordinator plan.

#### 14. Test 4: Tool failure

In [0]:
def mock_failed_sql_analytics_tool(
    user_request: str,
) -> Dict[str, Any]:
    return {
        "tool": "sql_analytics_tool",
        "status": "failed",
        "sql_action": None,
        "sql_result": None,
        "error": "Unsupported analytics request.",
    }


#### 15. Optional display helper

In [0]:
def display_sql_agent_result(
    result: SQLAgentResult,
) -> None:
    """
    Display the SQL Agent result in a readable format.
    """

    print("=" * 80)
    print("SQL AGENT RESULT")
    print("=" * 80)

    print(f"Agent: {result.agent_name}")
    print(f"Status: {result.status}")
    print(f"Message: {result.message}")
    print(f"Task: {result.task_description}")
    print(f"SQL Action: {result.sql_action}")
    print(f"Row Count: {result.row_count}")
    print(f"SQL Result: {result.sql_result}")
    print(f"Error: {result.error}")

In [0]:
display_sql_agent_result(
    state["agent_results"]["sql_agent"]
)

#### Part 3 key learnings

The SQL Agent:

- Does not classify the customer request.
- Does not decide whether SQL is required.
- Trusts the Coordinator’s execution plan.
- Runs only when assigned a task.
- Reuses the existing SQL Analytics Tool.
- Converts raw tool output into a validated SQLAgentResult.
- Stores its output in shared state.
- Records successful, failed, or skipped execution.
- Does not generate the final customer-facing answer.

The final response remains the responsibility of the Final Response Agent.


#### Part 3 conclusion


SQL Agent Responsibilities

- Follows the Coordinator Agent's execution plan.
- Executes only when a task is assigned to the SQL Agent.
- Calls the SQL Analytics Tool.
- Validates the tool output using the SQLAgentResult schema.
- Stores the validated result in the shared state's agent_results.
- Records its execution in the shared state's execution_history.
- Records an error in the shared state's errors list, if needed.
- Can be tested independently from the rest of the multi-agent workflow.


## Part 4 : Prediction Agent


Purpose
The Prediction Agent is responsible for executing prediction tasks assigned by the Coordinator Agent.

It does not decide prediction tool should be used.

It simply:

- Reads the execution plan.
- Checks whether it has been assigned a task.
- Executes the Prediction Tool.
- Validates the result.
- Updates the shared state.


#### Architecture

``` text

Shared State
      │
      ▼
Read Coordinator Result
      │
      ▼
Was prediction Agent assigned?
      │
   ┌──┴─────────┐
   │            │
  No           Yes
   │            │
   ▼            ▼
Record       Read assigned
Skipped      Prediction task
                │
                ▼
       Call Prediction tool
                │
                ▼
       Validate PredictionAgentResult
                │
                ▼
       Store in agent_results
                │
                ▼
       Record execution history
                │
                ▼
        Record error if needed


```



#### 1 : Imports

In [0]:
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from pydantic import BaseModel, Field, ValidationError


#### 2 : Execution-history schema

In [0]:
class ExecutionRecord(BaseModel):
    """
    Represents one agent execution event in the multi-agent workflow.
    """

    agent_name: AgentName

    status: AgentStatus

    message: str

    timestamp: datetime = Field(
        default_factory=lambda: datetime.now(timezone.utc)
    )

In [0]:
# Example: Create an execution record

ExecutionRecord(
    agent_name="prediction_agent",
    status="success",
    message="Prediction task completed successfully.",
)


#### 3 : Updated shared-state definition

In [0]:
class MultiAgentState(TypedDict):
    """
    Shared state used by all agents in the workflow.
    """

    user_request: str

    coordinator_result: Optional[CoordinatorResult]

    agent_results: Dict[str, BaseAgentResult]

    execution_history: List[ExecutionRecord]

    final_response: Optional[str]

    errors: List[Dict[str, Any]]


#### 4 : Initial-state function

In [0]:
def create_initial_state(
    user_request: str,
) -> MultiAgentState:
    """
    Create the initial shared state for one customer request.    
    """

    return {
        "user_request": user_request.strip(),
        "coordinator_result": None,
        "agent_results": {},
        "execution_history": [],
        "final_response": None,
        "errors": [],
    }

In [0]:
# Example

state = create_initial_state(
    "Video streaming keeps buffering"
)


#### 5 : Helper: find an assigned task

In [0]:
def get_agent_task(
    tasks: List[AgentTask],
    agent_name: AgentName,
) -> Optional[AgentTask]:
    """
    Return the task assigned to the requested agent.

    Parameters
    ----------
    tasks:
        Execution-plan tasks created by the Coordinator Agent.

    agent_name:
        Name of the specialist agent looking for its task.

    Returns
    -------
    Optional[AgentTask]
        The assigned task when found; otherwise None.
    """

    for task in tasks:
        if task.agent_name == agent_name:
            return task

    return None

In [0]:
# Example 1

prediction_task = get_agent_task(
    tasks=coordinator_result.tasks,
    agent_name="prediction_agent",
)

In [0]:
# Example 2

retention_task = get_agent_task(
    prediction_result.tasks,
    "retention_agent",
)

print(retention_task)


#### 6 : Helper -  record agent execution

In [0]:
# Helper Function: Record Workflow Errors

def record_agent_execution(
    state: MultiAgentState,
    agent_name: AgentName,
    status: AgentStatus,
    message: str,
) -> None:
    """
    Add one validated execution event to shared state.
    """

    execution_record = ExecutionRecord(
        agent_name=agent_name,
        status=status,
        message=message,
    )

    state["execution_history"].append(execution_record)

    # This helper modifies the shared-state dictionary directly, so it does not need to return the state.

In [0]:
state = create_initial_state(
    "Video streaming keeps buffering"
)

record_agent_execution(
    state,
    "prediction_agent",
    "success",
    "Prediction Agent executed successfully.",
)

print(state["execution_history"])

In [0]:
[
    {
        "agent_name": "prediction_agent",
        "status": "success",
        "message": "Prediction Agent executed successfully."
    }
]


#### 7 : Helper - record workflow errors

In [0]:
def record_agent_error(
    state: MultiAgentState,
    agent_name: AgentName,
    error_code: str,
    error_message: str,
) -> None:
    """
    Record one agent error in shared state.
    """

    state["errors"].append(
        {
            "agent_name": agent_name,
            "error_code": error_code,
            "error_message": error_message,
        }
    )

# This keeps error-handling code consistent across the specialist agents.


#### 8 : Prediction Agent function

In [0]:
def execute_prediction_agent(
    task: AgentTask,
    user_request: str,
) -> SQLAgentResult:
    """
    Execute the prediction Tool for one assigned predictionL task.

    The Prediction Agent does not decide whether Prediction is required.
    The Coordinator Agent has already made that decision.
    """

    try:
        tool_result = prediction_tool(user_request)

        if not isinstance(tool_result, dict):
            return SQLAgentResult(
                agent_name="sql_agent",
                status="failed",
                message="The SQL Analytics Tool returned an invalid result.",
                error="INVALID_SQL_TOOL_RESULT",
                task_description=task.task_description,
                sql_action=None,
                sql_result=None,
                row_count=None,
            )

        tool_status = tool_result.get("status")

        if tool_status != "success":
            return SQLAgentResult(
                agent_name="sql_agent",
                status="failed",
                message="The SQL Analytics Tool could not complete the request.",
                error=tool_result.get(
                    "error",
                    "SQL_TOOL_EXECUTION_FAILED",
                ),
                task_description=task.task_description,
                sql_action=tool_result.get("sql_action"),
                sql_result=tool_result.get("sql_result"),
                row_count=None,
            )

        sql_result = tool_result.get("sql_result")

        row_count = (
            len(sql_result)
            if isinstance(sql_result, list)
            else None
        )

        return SQLAgentResult(
            agent_name="sql_agent",
            status="success",
            message="SQL analytics task completed successfully.",
            error=None,
            task_description=task.task_description,
            sql_action=tool_result.get("sql_action"),
            sql_result=sql_result,
            row_count=row_count,
        )

    except Exception as exc:
        return SQLAgentResult(
            agent_name="sql_agent",
            status="failed",
            message="An unexpected error occurred in the SQL Agent.",
            error=str(exc),
            task_description=task.task_description,
            sql_action=None,
            sql_result=None,
            row_count=None,
        )